# FastAPI Integration with Railtracks

This notebook demonstrates how to build an agent using Railtracks and expose it as a RESTful web service using FastAPI.

By following this hands-on guide, you will learn to:
1. Define a tool and create an agent.
2. Wrap your agent in a `rt.Flow`.
3. Set up a FastAPI application running these agents asynchronously.
4. Send requests to the endpoint to test it.

In [1]:
!pip install railtracks fastapi uvicorn httpx nest_asyncio


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\App\Anaconda\python.exe -m pip install --upgrade pip


In [2]:
import nest_asyncio
nest_asyncio.apply()
import asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import railtracks as rt
import uvicorn
import httpx
import threading
import time
import os

## 1. Creating our Application Logic
First, we define a simple function tool, configure our LLM provider, and set up our railtracks `Agent` and `Flow`.

In [3]:
@rt.function_node
def weather_tool(location: str) -> str:
    """Gets the current weather for a specified location.
    Args:
        location: The city or region.
    Returns:
        String describing the weather.
    """
    return f"The weather in {location} is currently 72°F and sunny."

provider = rt.llm.GeminiLLM("gemini-3-flash-preview")
WeatherAgent = rt.agent_node(
    "WeatherAssistant", 
    tool_nodes=[weather_tool], 
    llm=provider,
    system_message="You are a helpful assistant that answers questions about the weather."
)

# Create the main Flow
weather_flow = rt.Flow(name="WeatherFlow", entry_point=WeatherAgent)

## 2. Defining the FastAPI Server
Now we'll map our `weather_flow` inside an async FastAPI route. We'll define schemas and hook it all up.

In [4]:
app = FastAPI(title="Railtracks Weather API")

class ChatRequest(BaseModel):
    query: str

class ChatResponse(BaseModel):
    response: str

@app.post("/chat", response_model=ChatResponse)
async def chat_endpoint(request: ChatRequest):
    try:
        # Execute our custom railtracks flow logic seamlessly inside FastAPI!
        result = await weather_flow.ainvoke(request.query)
        # Extract the content from the LLMResponse object
        return ChatResponse(response=str(result.content))
    except Exception as e:
        print(e)
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def healthcheck():
    return {"status": "ok"}

## 3. Running and Testing the Server
Let's launch the `uvicorn` server in a background thread so we can send requests to it directly inside this notebook.

In [5]:
server = None

def run_server():
    global server
    # Standard setup for uvicorn programmatic running.
    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="error")
    server = uvicorn.Server(config)
    server.run()

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Give it a second to spin up
time.sleep(2)

Let's send an HTTP request to health-check the endpoint.

In [6]:
with httpx.Client() as client:
    resp = client.get("http://127.0.0.1:8000/health")
    print("Health Status:", resp.json())

Health Status: {'status': 'ok'}


Let's hit the main `/chat` endpoint with a query that our agent will process along with tools.

In [7]:
payload = {
    "query": "What is the weather like in Seattle today?"
}

with httpx.Client(timeout=30.0) as client:
    # Using httpx to make an actual API call to FastAPI webserver
    resp = client.post("http://127.0.0.1:8000/chat", json=payload)
    print("Agent API Response:")
    print(resp.json()['response'])

Agent API Response:
OK. The weather in Seattle is currently 72°F and sunny.


## Wrap up

This displays how naturally Railtracks `Flow` objects execute within standard asynchronous web applications. By leaning natively on standard asynchronous execution, you never have to resort to hacky callback logic when serving your Agents. You can inject external contexts gracefully using endpoint HTTP headers as needed.

In [8]:
# close the socket
if server:
    server.should_exit = True
    server_thread.join()
print("Server stopped.")

Server stopped.
